# BM25

In [ ]:
import fitz 

pdf_files = ["ToyData/test.pdf", "ToyData/Natural-Language-Processing-Python.pdf", "ToyData/learning-langchain-for-true-epub-9781098167288.pdf"]
corpus = []
for pdf in pdf_files:
    doc = fitz.open(pdf)
    text = ""
    for page in doc:
        text += page.get_text()
    corpus.append(text)

In [27]:
import re

STOPWORDS = set(
    "a an the is are was were be been being and or of to in on for with as it its this that "
    "what how when where which who do does did can could will would should i you he she they we "
    "at by from not no".split()
)

def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]

In [28]:
query1 = "What is Tokenization?"
query2 = "What are different types of tokenization?"

In [29]:
tokenized_corpus = [tokenize(doc) for doc in corpus]
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

tokenized_query = tokenize(query1)

scores = bm25.get_scores(tokenized_query)

for pdf, score in zip(pdf_files, scores):
    print(pdf, score)

best_index = scores.argmax()

print("Most relevant PDF:", pdf_files[best_index])

ToyData/test.pdf -0.022513493395923328
ToyData/Natural-Language-Processing-Python.pdf -0.021823806103995636
ToyData/learning-langchain-for-true-epub-9781098167288.pdf -0.011668281583320337
Most relevant PDF: ToyData/learning-langchain-for-true-epub-9781098167288.pdf


In [30]:
for pdf, score in zip(pdf_files, scores):
    print(f"Document {pdf} -> BM25 Score: {score:.4f}")

Document ToyData/test.pdf -> BM25 Score: -0.0225
Document ToyData/Natural-Language-Processing-Python.pdf -> BM25 Score: -0.0218
Document ToyData/learning-langchain-for-true-epub-9781098167288.pdf -> BM25 Score: -0.0117


# By extracting Pages

In [31]:
import fitz
from rank_bm25 import BM25Okapi

pdf_files = ["ToyData/test.pdf", "ToyData/Natural-Language-Processing-Python.pdf", "ToyData/learning-langchain-for-true-epub-9781098167288.pdf"]

pages = []
page_info = []

# Extract each page as one document
for pdf in pdf_files:
    doc = fitz.open(pdf)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        if text.strip():
            pages.append(text)
            page_info.append((pdf, page_num))

In [32]:
query2 = "What are different types of tokenization?"

In [33]:
tokenized_pages = [tokenize(page) for page in pages]
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(tokenized_pages)


In [34]:
tokenized_query = tokenize(query2)
scores = bm25.get_scores(tokenized_query)

In [35]:
import numpy as np
for idx in np.argsort(scores)[::-1][:5]:
    pdf_name, page_num = page_info[idx]
    print(f"{pdf_name} (page {page_num}) -> {scores[idx]:.4f}")

ToyData/test.pdf (page 75) -> 7.6962
ToyData/test.pdf (page 31) -> 7.3033
ToyData/test.pdf (page 93) -> 7.0984
ToyData/test.pdf (page 77) -> 7.0815
ToyData/test.pdf (page 360) -> 7.0428


In [36]:
top_indices = scores.argsort()[::-1][:5]

for idx in top_indices:
    pdf_name, page_num = page_info[idx]
    print("PDF:", pdf_name)
    print("Page:", page_num)
    print("Score:", scores[idx])
    print("Text Preview:", pages[idx][:300])
    print("-" * 60)

PDF: ToyData/test.pdf
Page: 75
Score: 7.696192316899927
Text Preview: Third, the tokenizer needs to be trained on a specific dataset to establish the
best vocabulary it can use to represent that dataset. Even if we set the same
methods and parameters, a tokenizer trained on an English text dataset will
be different from another trained on a code dataset or a multiling
------------------------------------------------------------
PDF: ToyData/test.pdf
Page: 31
Score: 7.3033380942995
Text Preview: properties make sense to a computer and serve as a good way to translate
human language into computer language.
Embeddings are tremendously helpful as they allow us to measure the
semantic similarity between two words. Using various distance metrics, we
can judge how close one word is to another. As
------------------------------------------------------------
PDF: ToyData/test.pdf
Page: 93
Score: 7.098413593433634
Text Preview: Aside from these, the LLM designer can add tokens that help better
m

# Now by creating chunks 

In [37]:
import fitz
from rank_bm25 import BM25Okapi

pdf_files = ["ToyData/test.pdf", "ToyData/Natural-Language-Processing-Python.pdf", "ToyData/learning-langchain-for-true-epub-9781098167288.pdf"]

def create_chunks(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks

all_chunks = []
chunk_info = []

for pdf in pdf_files:
    doc = fitz.open(pdf)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        chunks = create_chunks(text, chunk_size=150, overlap=30)

        for chunk_id, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_info.append({
                "pdf": pdf,
                "page": page_num,
                "chunk_id": chunk_id,
                "text": chunk
            })

print("Total chunks:", len(all_chunks))

Total chunks: 3793


In [38]:
tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

In [39]:
query = "What is Tokenization in NLP?"
tokenized_query = tokenize(query)
scores = bm25.get_scores(tokenized_query)

In [40]:
import numpy as np
for idx in np.argsort(scores)[::-1][:5]:
    info = chunk_info[idx]
    print(f"{info['pdf']} (p.{info['page']}, chunk {info['chunk_id']}) -> {scores[idx]:.4f}")

ToyData/Natural-Language-Processing-Python.pdf (p.100, chunk 2) -> 7.9894
ToyData/test.pdf (p.76, chunk 0) -> 7.9025
ToyData/test.pdf (p.75, chunk 1) -> 7.3470
ToyData/test.pdf (p.91, chunk 1) -> 6.9147
ToyData/Natural-Language-Processing-Python.pdf (p.65, chunk 0) -> 6.7699


In [41]:
import numpy as np
for idx in np.argsort(scores)[::-1][:5]:
    info = chunk_info[idx]
    print(f"PDF: {info['pdf']}")
    print(f"Page: {info['page']}")
    print(f"Chunk ID: {info['chunk_id']}")
    print(f"BM25 Score: {scores[idx]:.4f}")
    print('-'*50)

PDF: ToyData/Natural-Language-Processing-Python.pdf
Page: 100
Chunk ID: 2
BM25 Score: 7.9894
--------------------------------------------------
PDF: ToyData/test.pdf
Page: 76
Chunk ID: 0
BM25 Score: 7.9025
--------------------------------------------------
PDF: ToyData/test.pdf
Page: 75
Chunk ID: 1
BM25 Score: 7.3470
--------------------------------------------------
PDF: ToyData/test.pdf
Page: 91
Chunk ID: 1
BM25 Score: 6.9147
--------------------------------------------------
PDF: ToyData/Natural-Language-Processing-Python.pdf
Page: 65
Chunk ID: 0
BM25 Score: 6.7699
--------------------------------------------------


In [42]:
import numpy as np

top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

for idx in top_indices:
    print(f"PDF: {chunk_info[idx]['pdf']}")
    print(f"Page: {chunk_info[idx]['page']}")
    print(f"Chunk ID: {chunk_info[idx]['chunk_id']}")
    print(f"BM25 Score: {scores[idx]:.4f}")
    print("\nRelevant Chunk:")
    print(all_chunks[idx])
    print("=" * 100)

PDF: ToyData/Natural-Language-Processing-Python.pdf
Page: 100
Chunk ID: 2
BM25 Score: 7.9894

Relevant Chunk:
to connect negation words (such as “not” or “never”) to the positive words they might be used to qualify. We leave it to you to continue the NLP action by improving on this machine learn- ing model. And you can check your progress relative to VADER at each step of the way to see if you think machine learning is a better approach than hard-coding algorithms for NLP. Summary You implemented tokenization and configured a tokenizer for your application. n-gram tokenization helps retain some of the word order information in a document. Your new bags of words have some tokens that weren’t in the original bags of words DataFrame (23302 columns now instead of 20756 before). You need to make sure your new product DataFrame of bags of words has the exact same columns (tokens) in the exact same order as the original one used to train your Naive Bayes
PDF: ToyData/test.pdf
Page: 76
Chunk